# Extract SMILES Strings from geom_drugs_boltz Dataset

This notebook extracts all SMILES strings from the processed geom_drugs_boltz dataset.


In [ ]:
import json
from pathlib import Path
import random
from rdkit import Chem

# Dataset path
# DATASET_DIR = Path("/datastor1/dy4652/proteinzen/geom_drugs_conformers")
DATASET_DIR = Path("/datastor1/dy4652/proteinzen/geom_drugs_boltz")
MANIFEST_PATH = DATASET_DIR / "manifest.json"




In [3]:
# Load the manifest
with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f"Total entries in manifest: {len(manifest)}")


Total entries in manifest: 1000


In [9]:
entry['structures']

[{'resolution': None,
  'method': 'QM9:OC(CNC1CCCCC1)C(F)(F)F',
  'deposited': None,
  'released': None,
  'revised': None,
  'num_chains': 1,
  'num_interfaces': 0,
  'pH': None,
  'temperature': None},
 {'resolution': None,
  'method': 'QM9:OC(CNC1CCCCC1)C(F)(F)F',
  'deposited': None,
  'released': None,
  'revised': None,
  'num_chains': 1,
  'num_interfaces': 0,
  'pH': None,
  'temperature': None},
 {'resolution': None,
  'method': 'QM9:OC(CNC1CCCCC1)C(F)(F)F',
  'deposited': None,
  'released': None,
  'revised': None,
  'num_chains': 1,
  'num_interfaces': 0,
  'pH': None,
  'temperature': None},
 {'resolution': None,
  'method': 'QM9:OC(CNC1CCCCC1)C(F)(F)F',
  'deposited': None,
  'released': None,
  'revised': None,
  'num_chains': 1,
  'num_interfaces': 0,
  'pH': None,
  'temperature': None},
 {'resolution': None,
  'method': 'QM9:OC(CNC1CCCCC1)C(F)(F)F',
  'deposited': None,
  'released': None,
  'revised': None,
  'num_chains': 1,
  'num_interfaces': 0,
  'pH': None,
  't

In [ ]:
# Extract SMILES strings
smiles_list = []
failed = 0

for entry in manifest:
    try:
        method = entry.get('structure', {}).get('method', '')
        if method.startswith('QM9:'):
            smiles = method[4:]  # Remove 'QM9:' prefix
            smiles_list.append(smiles)
        else:
            failed += 1
    except Exception as e:
        failed += 1

print(f"Extracted {len(smiles_list)} SMILES strings")
print(f"Failed to extract: {failed} entries")
print(f"\nFirst 5 SMILES:")
for i, smiles in enumerate(smiles_list[:5], 1):
    print(f"  {i}. {smiles[:80]}..." if len(smiles) > 80 else f"  {i}. {smiles}")

geom_drugs_set = set(smiles_list)


Extracted 1000 SMILES strings
Failed to extract: 0 entries

First 5 SMILES:
  1. CC(C)CCC(C(=O)O)C(C)CC(=O)NC1CCCCC1
  2. CCCN(CCC)CCCNC(=O)c1ccc2c(Cl)c3c(nc2c1)CCC3
  3. NCCS(=O)(=O)[NH2+]C1CCCCC1
  4. CCCCNC(=O)COC(=O)c1c2c(nc3ccccc13)CCC2
  5. CC(C)C[NH2+]c1c2c(nc3ccccc13)CCC2


In [11]:
num_heavy_atoms_list = []
for smi in geom_drugs_set:
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        num_heavy_atoms = mol.GetNumHeavyAtoms()
        num_heavy_atoms_list.append(num_heavy_atoms)
    

In [13]:
max(num_heavy_atoms_list)

36

In [14]:
# Some basic statistics
from collections import Counter

print("SMILES Statistics:")
print("=" * 60)
print(f"Total unique SMILES: {len(set(smiles_list))}")
print(f"Total SMILES (with duplicates): {len(smiles_list)}")
print(f"Number of duplicates: {len(smiles_list) - len(set(smiles_list))}")

# Check for duplicate SMILES
smiles_counts = Counter(smiles_list)
duplicates = {smiles: count for smiles, count in smiles_counts.items() if count > 1}
if duplicates:
    print(f"\nFound {len(duplicates)} SMILES that appear multiple times:")
    for smiles, count in list(duplicates.items())[:5]:
        print(f"  '{smiles[:60]}...' appears {count} times")
else:
    print("\n✓ No duplicate SMILES found (each SMILES appears exactly once)")


SMILES Statistics:
Total unique SMILES: 1000
Total SMILES (with duplicates): 1000
Number of duplicates: 0

✓ No duplicate SMILES found (each SMILES appears exactly once)


In [15]:
# from openqdc.datasets import QMugs
# from pathlib import Path

# # Set the cache directory directly as a parameter (this is the proper way)
# # The cache_dir parameter is the correct way to set the cache location
# cache_dir = Path("/datastor1/dy4652/proteinzen/openqdc_cache")
# cache_dir.mkdir(parents=True, exist_ok=True)

# # Initialize QMugs with cache_dir parameter
# dataset = QMugs(
#     energy_unit="kcal/mol",
#     distance_unit="ang",
#     array_format="torch",
#     cache_dir=str(cache_dir)  # Pass cache_dir as a parameter instead of using env var
# )

In [16]:
# Access SMILES from QMugs dataset
# SMILES are stored in dataset.data['name']
# Each entry in 'name' is a SMILES string (one per conformer)
# import numpy as np

# print("Accessing SMILES from QMugs dataset:")
# print("=" * 60)

# # Check what's in the data dictionary
# if hasattr(dataset, 'data') and dataset.data is not None:
#     print(f"\nData keys: {list(dataset.data.keys())}")
    
#     # Access SMILES - they're stored in the 'name' field
#     smiles_array = dataset.data['name']
#     print(f"\nSMILES array shape: {smiles_array.shape}")
#     print(f"SMILES array dtype: {smiles_array.dtype}")
#     print(f"Total number of SMILES (conformers): {len(smiles_array)}")
    
#     # Show first few SMILES
#     print(f"\nFirst 5 SMILES:")
#     for i in range(min(5, len(smiles_array))):
#         smiles_str = str(smiles_array[i])
#         print(f"  {i+1}. {smiles_str[:80]}..." if len(smiles_str) > 80 else f"  {i+1}. {smiles_str}")
    
#     # Get unique SMILES (since there are multiple conformers per molecule)
#     # Convert to Python set of regular strings
#     unique_smiles_set = {str(s) for s in np.unique(smiles_array)}
#     print(f"\nUnique SMILES (molecules): {len(unique_smiles_set)}")
#     print(f"Total conformers: {len(smiles_array)}")
#     print(f"Average conformers per molecule: {len(smiles_array) / len(unique_smiles_set):.2f}")
#     print(f"Type of unique_smiles_set: {type(unique_smiles_set)}")
#     print(f"Element type: {type(next(iter(unique_smiles_set)))}")
# else:
#     print("Dataset data not available yet. Make sure dataset is fully loaded.")

In [18]:
# # Extract all unique SMILES from QMugs dataset
# # Get unique SMILES (removes duplicates from multiple conformers)
# if hasattr(dataset, 'data') and dataset.data is not None:
#     qmugs_smiles = np.unique(dataset.data['name'])
    
#     print(f"Extracted {len(qmugs_smiles)} unique SMILES from QMugs dataset")
#     print(f"\nFirst 10 unique SMILES:")
#     for i, smiles in enumerate(qmugs_smiles[:10], 1):
#         smiles_str = str(smiles)
#         print(f"  {i}. {smiles_str[:70]}..." if len(smiles_str) > 70 else f"  {i}. {smiles_str}")
    
# else:
#     print("Dataset data not available. Make sure dataset is fully loaded.")


In [ ]:
# Convert to a Python set of regular strings
# If qmugs_smiles is a numpy array, convert each element to a regular Python string
# qmugs_smiles_set = {str(s) for s in qmugs_smiles} if 'qmugs_smiles' in locals() else set()

In [19]:
# import random
# random.seed(42)
# qmugs_smiles_sample = random.sample(qmugs_smiles_set, 30)

In [20]:
# from rdkit import Chem

# def canonicalize_smiles(smiles_list, isomeric=True):
#     canon_to_noncanon = dict()
#     for s in smiles_list:
#         mol = Chem.MolFromSmiles(s)
#         if mol is None:
#             continue
#         else:
#             canon_to_noncanon[(Chem.MolToSmiles(mol, canonical=True, isomericSmiles=isomeric))] = s
#     return canon_to_noncanon


# canon_qmugs_smiles_dict = canonicalize_smiles(qmugs_smiles_set)
# canon_geom_drugs_dict = canonicalize_smiles(geom_drugs_set)


In [21]:
# import random

# random.seed(42)

# canon_qmugs_smiles_set = set(canon_qmugs_smiles_dict.keys())
# canon_geom_drugs_set = set(canon_geom_drugs_dict.keys())

# ood_set = canon_qmugs_smiles_set - canon_geom_drugs_set
# len(canon_qmugs_smiles_set), len(ood_set)

# ood_set_sample = random.sample(ood_set, 30)

In [ ]:
# Create YAML file in geom_sampling.yaml format from ood_set_sample
import yaml

# Use ood_set_sample (30 SMILES)
# Convert to list of strings if it's a set or list
smiles_to_use = sorted([str(s) for s in ood_set_sample])

# Create tasks list in the required format
tasks = []
for smiles in smiles_to_use:
    tasks.append({
        "task": "unconditional_smiles",
        "smiles": smiles,
        "num_samples": 1
    })

# Create the YAML structure
yaml_data = {
    "tasks": tasks
}

# Write to YAML file
output_yaml = Path("sampling/geom_sampling.yaml")
with open(output_yaml, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Created YAML file with {len(tasks)} tasks from ood_set_sample")
print(f"Saved to: {output_yaml.absolute()}")
print(f"\nAll {len(tasks)} SMILES in the file:")
for i, task in enumerate(tasks, 1):
    print(f"  {i}. {task['smiles']}")


✓ Created YAML file with 30 tasks from ood_set_sample
✓ Saved to: /datastor1/dy4652/proteinzen/sampling/geom_sampling.yaml

All 30 SMILES in the file:
  1. CC(C)(C)C(=O)N1CCC(n2c(=O)[nH]c3ccccc32)CC1
  2. CC(C)Cn1ncc2cc(Oc3ccc(F)cc3F)c(C(=O)NCCN(C)C)cc21
  3. CC(C)Oc1ccc(Nc2nc(=O)n(Cc3ccccn3)c(=O)n2Cc2ccc(Cl)cc2)cc1F
  4. CC1(C)C=Cc2c(cc(O)c3c(=O)c4cccc(O)c4[nH]c23)O1
  5. CC1(C)OC(=O)C(OC2CCCC2)=C1c1ccc(S(C)(=O)=O)cc1
  6. CC1=CC[C@@H](NS(=O)(=O)c2ccc(Oc3ccc(Cl)cc3)cc2)C(=O)N(O)C1
  7. CCNC(=S)Nc1cc(-n2cnc(C)c2)ncn1
  8. CCOC(=O)CCCc1sc(N)nc1-c1[nH]c(O)nc1C
  9. CCOC(=O)c1cc(Nc2ncc(Cl)c(-c3cccc(CC#N)c3)n2)ccc1N1CCN(C)CC1
  10. CC[C@@H](N[C@H](C)c1cnc[nH]1)c1ccc(Cl)c(C(=O)c2cccnc2)c1F
  11. CN1CCN(C(=O)c2cc3cccc(F)c3[nH]2)CC1
  12. CN1C[C@@H](C(=O)NCc2c(Cl)cc(Cl)cc2Cl)N(C)C1=O
  13. CNCCC[C@@H](c1ccc(Cl)c(Cl)c1)c1cccc2[nH]ccc12
  14. CNc1cc(Nc2ccccc2OC)nn2c(C(=O)NCC(C)(C)CO)cnc12
  15. COc1cc(OCCc2ccc(Cl)cc2Cl)cc(C(=O)NCC2CCN(c3ccncc3)CC2)c1
  16. COc1ccc2c3c1O[C@H]1CC=C[C@H]4[C@@H](C2

In [15]:
# Write QMugs conformers to PDB files using RDKit directly
import numpy as np
from pathlib import Path
from rdkit import Chem
from rdkit.Chem.rdchem import RWMol, Conformer
from rdkit.Geometry import Point3D

def write_qmugs_conformer_to_pdb(smiles: str, atomic_inputs, output_path: Path):
    """
    Write QMugs conformer to PDB using RDKit directly.

    Parameters
    ----------
    smiles : str
        SMILES string (USED ONLY AS AN IDENTIFIER; NOT FOR CHEMISTRY)
    atomic_inputs : np.ndarray
        Shape (N, 5): [atomic_number, formal_charge, x, y, z]
    output_path : Path
        Output PDB path
    """

    atomic_inputs = np.asarray(atomic_inputs)

    Z = atomic_inputs[:, 0].astype(int)
    q = atomic_inputs[:, 1].astype(int)
    xyz = atomic_inputs[:, 2:5].astype(float)

    n_atoms = len(Z)

    mol = RWMol()
    conf = Conformer(n_atoms)

    for i in range(n_atoms):
        atom = Chem.Atom(int(Z[i]))
        atom.SetFormalCharge(int(q[i]))
        idx = mol.AddAtom(atom)
        conf.SetAtomPosition(idx, Point3D(*xyz[i]))

    mol = mol.GetMol()
    conf.SetId(0)
    mol.AddConformer(conf, assignId=True)

    from rdkit.Chem import rdchem

    
    for atom in mol.GetAtoms():
        atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)

    for bond in mol.GetBonds():
        bond.SetStereo(rdchem.BondStereo.STEREONONE)


    output_path.parent.mkdir(parents=True, exist_ok=True)
    Chem.MolToPDBFile(mol, str(output_path), confId=0)

    return True


def write_qmugs_smiles_list_to_pdb(
    smiles_list: list,
    qmugs_dataset,
    output_dir: Path,
    conformer_idx: int = 0,
):
    """
    Batch process a list of SMILES and write QMugs conformers to PDB files.

    IMPORTANT:
    - SMILES is ONLY a lookup key
    - Geometry comes ONLY from QMugs atomic_inputs
    """

    data = qmugs_dataset.data
    output_dir.mkdir(parents=True, exist_ok=True)

    # Build SMILES -> conformer indices map
    smiles_array = data["name"]
    smiles_to_indices = {}

    for i, s in enumerate(smiles_array):
        s = str(s)
        smiles_to_indices.setdefault(s, []).append(i)

    results = {}

    for i, smiles in enumerate(smiles_list):
        smiles = str(smiles)

        if smiles not in smiles_to_indices:
            print(f"❌ SMILES not found in QMugs: {smiles[:60]}...")
            results[smiles] = None
            continue

        indices = smiles_to_indices[smiles]

        if conformer_idx >= len(indices):
            print(
                f"❌ Conformer index {conformer_idx} out of range for "
                f"{smiles[:60]}... (found {len(indices)})"
            )
            results[smiles] = None
            continue

        idx = indices[conformer_idx]
        lo, hi = data["position_idx_range"][idx]
        atomic_inputs = data["atomic_inputs"][lo:hi]

        safe_name = (
            smiles.replace("/", "_")
                  .replace("\\", "_")
                  .replace(":", "_")
                  .replace("*", "_")
        )[:60]

        output_path = output_dir / f"{i}.pdb"

        try:
            write_qmugs_conformer_to_pdb(
                smiles=smiles,              # ignored internally
                atomic_inputs=atomic_inputs,
                output_path=output_path,
            )
            results[smiles] = output_path
            # print(f"✓ Wrote {output_path} ({len(atomic_inputs)} atoms)")
        except Exception as e:
            print(f"Failed for {smiles[:60]}...")
            import traceback
            traceback.print_exc()
            results[smiles] = None

    return results


In [16]:
# smiles_list = sorted([canon_qmugs_smiles_dict[s] for s in ood_set_sample])
# smiles_list = sorted([canon_qmugs_smiles_dict[s] for s in ood_set_sample])
output_dir = Path("sampling/qmugs_ref")
smiles_list = sorted(qmugs_smiles_sample)

print(f"Processing {len(smiles_list)} SMILES from QMugs...")
results = write_qmugs_smiles_list_to_pdb(
    smiles_list=smiles_list,
    qmugs_dataset=dataset,
    output_dir=output_dir,
    conformer_idx=0  # Use first conformer
)

# Summary
successful = sum(1 for v in results.values() if v is not None)
print(f"\nSuccessfully wrote {successful}/{len(smiles_list)} PDB files to {output_dir}")


Processing 30 SMILES from QMugs...

Successfully wrote 30/30 PDB files to sampling/qmugs_ref


In [ ]:
import yaml
# Create tasks list in the required format
tasks = []
output_dir = Path("sampling/qmugs_ref")
smiles_list = sorted(qmugs_smiles_sample)

for i, smiles in enumerate(smiles_list):
    tasks.append({
        "task": "unconditional_mol",
        "mol_pdb_path": str(Path(output_dir / f"{i}.pdb").resolve()),
        "num_samples": 5,
        "name": f"sample_{i}"
    })

# Create the YAML structure``
yaml_data = {
    "tasks": tasks
}

# Write to YAML file
output_yaml = Path("sampling/geom_sampling_qmugs_test_mol.yaml")
output_yaml.parent.mkdir(parents=True, exist_ok=True)
with open(output_yaml, 'w') as f:
    yaml.dump(yaml_da ta, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Created YAML file with {len(tasks)} tasks from geom_drugs_sample")
print(f"Saved to: {output_yaml.absolute()}")


Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_sampling_qmugs_test_mol.yaml


In [20]:
random.seed(42)
geom_drugs_sample = random.sample(sorted(geom_drugs_set), 30)

In [8]:
# Write geom_drugs_sample SMILES to YAML file
import yaml

# Convert to sorted list of strings
smiles_to_use = sorted([str(s) for s in geom_drugs_sample])

# Create tasks list in the required format
tasks = []
for i, smiles in enumerate(smiles_to_use):
    tasks.append({
        "task": "unconditional_smiles",
        "smiles": smiles,
        "num_samples": 3,
        "name": f"sample_{i}"
    })

# Create the YAML structure``
yaml_data = {
    "tasks": tasks
}

# Write to YAML file
output_yaml = Path("sampling/geom_sampling_train.yaml")
output_yaml.parent.mkdir(parents=True, exist_ok=True)
with open(output_yaml, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Created YAML file with {len(tasks)} tasks from geom_drugs_sample")
print(f"Saved to: {output_yaml.absolute()}")


Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_sampling_train.yaml


In [6]:
# Write geom_drugs_sample molecules to PDB files
import numpy as np
from rdkit import Chem
from rdkit.Chem.rdchem import RWMol, Conformer
from rdkit.Geometry import Point3D

def get_molecule_id_from_smiles(smiles: str, manifest: list):
    """Find molecule ID from SMILES in manifest."""
    for entry in manifest:
        method = entry.get('structure', {}).get('method', '')
        if method.startswith('QM9:'):
            entry_smiles = method[4:]  # Remove 'QM9:' prefix
            if entry_smiles == smiles:
                return entry['id']
    return None

def get_npz_path(molecule_id: str, dataset_dir: Path) -> Path:
    """Get path to .npz file from molecule ID."""
    # Subdirectory is characters at index 1-2 of the ID
    subdir = molecule_id[1:3]
    return dataset_dir / "structures" / subdir / f"{molecule_id}.npz"

def write_geom_drugs_molecule_to_pdb(smiles: str, npz_path: Path, output_path: Path):
    """Write geom_drugs_boltz molecule to PDB using RDKit."""
    # Load the .npz file
    data = np.load(npz_path)
    atoms = data['atoms']
    
    # Extract data
    Z = atoms['element'].astype(int)  # Atomic numbers
    q = atoms['charge'].astype(int)   # Formal charges
    xyz = atoms['coords'].astype(float)  # Coordinates
    
    n_atoms = len(Z)
    
    # Create RDKit molecule
    mol = RWMol()
    conf = Conformer(n_atoms)
    
    for i in range(n_atoms):
        atom = Chem.Atom(int(Z[i]))
        atom.SetFormalCharge(int(q[i]))
        idx = mol.AddAtom(atom)
        conf.SetAtomPosition(idx, Point3D(*xyz[i]))
    
    mol = mol.GetMol()
    conf.SetId(0)
    mol.AddConformer(conf, assignId=True)
    
    # Clear chirality and stereo (similar to QMugs function)
    from rdkit.Chem import rdchem
    for atom in mol.GetAtoms():
        atom.SetChiralTag(rdchem.ChiralType.CHI_UNSPECIFIED)
    for bond in mol.GetBonds():
        bond.SetStereo(rdchem.BondStereo.STEREONONE)
    
    # Write to PDB
    output_path.parent.mkdir(parents=True, exist_ok=True)
    Chem.MolToPDBFile(mol, str(output_path), confId=0)
    
    return True

# Build SMILES -> molecule ID mapping from manifest
print("Building SMILES to molecule ID mapping...")
smiles_to_id = {}
for entry in manifest:
    method = entry.get('structure', {}).get('method', '')
    if method.startswith('QM9:'):
        entry_smiles = method[4:]
        smiles_to_id[entry_smiles] = entry['id']

print(f"Mapped {len(smiles_to_id)} SMILES to molecule IDs\n")

# Process each SMILES in geom_drugs_sample
output_dir = Path("sampling/geom_drugs_ref")
output_dir.mkdir(parents=True, exist_ok=True)

results = {}
for i, smiles in enumerate(smiles_to_use):
    smiles = str(smiles)
    
    if smiles not in smiles_to_id:
        print(f"❌ SMILES not found in manifest: {smiles[:60]}...")
        results[smiles] = None
        continue
    
    molecule_id = smiles_to_id[smiles]
    npz_path = get_npz_path(molecule_id, DATASET_DIR)
    
    if not npz_path.exists():
        print(f"❌ Structure file not found: {npz_path}")
        results[smiles] = None
        continue
    
    output_path = output_dir / f"{i}.pdb"
    
    try:
        write_geom_drugs_molecule_to_pdb(
            smiles=smiles,
            npz_path=npz_path,
            output_path=output_path
        )
        results[smiles] = output_path
        # print(f"✓ Wrote {output_path.name} ({len(np.load(npz_path)['atoms'])} atoms)")
    except Exception as e:
        print(f"Failed for {smiles[:60]}...")
        import traceback
        traceback.print_exc()
        results[smiles] = None

# Summary
successful = sum(1 for v in results.values() if v is not None)
print(f"\n{'='*60}")
print(f"Successfully wrote {successful}/{len(geom_drugs_sample)} PDB files to {output_dir}")


Building SMILES to molecule ID mapping...
Mapped 304338 SMILES to molecule IDs


Successfully wrote 30/30 PDB files to sampling/geom_drugs_ref


In [8]:
# Write geom_drugs_sample SMILES to YAML file
import yaml
output_dir = Path("sampling/geom_drugs_ref")
output_dir.mkdir(parents=True, exist_ok=True)

# Convert to sorted list of strings
smiles_to_use = sorted([str(s) for s in geom_drugs_sample])

# Create tasks list in the required format
tasks = []
for i, smiles in enumerate(smiles_to_use):
    tasks.append({
        "task": "unconditional_mol",
        "mol_pdb_path": str(Path(output_dir / f"{i}.pdb").resolve()),
        "num_samples": 5,
        "name": f"sample_{i}"
    })

# Create the YAML structure``
yaml_data = {
    "tasks": tasks
}

# Write to YAML file
output_yaml = Path("sampling/geom_sampling_train_mol.yaml")
output_yaml.parent.mkdir(parents=True, exist_ok=True)
with open(output_yaml, 'w') as f:
    yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False, allow_unicode=True)

print(f"Created YAML file with {len(tasks)} tasks from geom_drugs_sample")
print(f"Saved to: {output_yaml.absolute()}")


Created YAML file with 30 tasks from geom_drugs_sample
Saved to: /datastor1/dy4652/proteinzen/sampling/geom_sampling_train_mol.yaml
